In [1]:
import os
os.environ['LOGS_DIRECTORY'] = '../eval_logs'

import sys
sys.path.insert(0, '..')

In [2]:
import json
import random

from tqdm.auto import tqdm
from pydantic import BaseModel
from pydantic_ai import Agent

from ingest import read_repo_data, chunk_documents
from main import initialize_index, initialize_agent
from logs import log_interaction_to_file, LOG_DIR

In [3]:
REPO_OWNER = "huggingface"
REPO_NAME = "pytorch-image-models"

In [4]:
docs = read_repo_data(REPO_OWNER, REPO_NAME)
pytorch_img_models_chunks = chunk_documents(docs, 2000, 1000)
index, vindex = initialize_index()
agent = initialize_agent(index, vindex)

Starting AI FAQ Assistant for huggingface/pytorch-image-models
Initializing data ingestion...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Data indexing completed successfully!
Initializing search agent...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Agent initialized successfully!


In [5]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about the PyTorch Image Models (timm) repository.

Based on the provided repository content (documentation, model descriptions, and code snippets), generate realistic questions that users (developers, ML engineers, or researchers) might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.

IMPORTANT:
- Questions should be answerable using the provided content
- Do NOT invent information outside the repository
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='gpt-4o-mini',
    output_type=QuestionsList
)

In [6]:
sample = random.sample(pytorch_img_models_chunks, 7)
prompt_docs = [
    {
        "title": d.get("title", ""),
        "content": d.get("chunk", ""),
        "source": d.get("filename", "")
    }
    for d in sample
]
prompt = f"""
Generate questions based on the following repository content:

{json.dumps(prompt_docs, indent=2)}
"""

result = await question_generator.run(prompt)
questions = result.output.questions

In [7]:
for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()

  0%|          | 0/6 [00:00<?, ?it/s]

What should I follow for training a new model afresh in the timm repository?
To train a new model afresh in the timm repository, you should follow the instructions outlined in the [timm recipe scripts](https://github.com/rwightman/pytorch-image-models/blob/master/train.py). These scripts contain the necessary methods and examples to help you set up your training process. 

Here is a brief outline of the steps you can follow:

1. **Select Your Model**: Choose a model architecture from the timm repository. You can find available models and their details within the documentation.

2. **Prepare Your Dataset**: Ensure that your dataset is formatted correctly and is ready for loading.

3. **Use the Training Script**: Adapt the training script provided in the timm repository. You can include any specific settings like learning rate, number of epochs, etc., based on your needs.

4. **Run Your Training Loop**: Execute the script to start training your model.

You can start from the main trainin